# MTHexapod Faults per strut

As preparation for the shutdown that will happen on Sep 2025,  
we want to find out what it the strut that is responsible for the major number of faults.  

Relevant documents:
  - [SITCOM-2192 Hexapod Faults per Strut](https://ls.st/SITCOM-2192)
  - [Copley Drives Manual](https://copleycontrols.com/wp-content/uploads/2018/02/All-CANopen_Programmers_Manual-Manual.pdf)
  - [MTHexapod - mthexapod.electrical](https://ts-xml.lsst.io/sal_interfaces/MTHexapod.html#electrical)

The `sal_index` can be either 1 (Camera Hexapod) or 2 (M2 Hexapod).  
The `min_log_level` correspondts to the minimum logging level required to print the messages.  

In [ ]:
start_day_obs = 20250801  # First light
end_day_obs = 20250825  # Start of maintenance shutdown
sal_index = 2
min_log_level = 30

In [ ]:
import enum
import ipywidgets as w
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

from astropy.time import Time
from datetime import timedelta
from IPython.display import display
from html import escape
from types import SimpleNamespace
from typing import Iterable, List, Tuple

from bokeh.plotting import figure
from bokeh.models import (
    ColumnDataSource, CDSView, BooleanFilter, HoverTool, FactorRange, Legend, Span, CustomJS
)
from bokeh.layouts import gridplot

from lsst_efd_client import EfdClient
from lsst.summit.utils.efdUtils import (
    getEfdData,
    getDayObsEndTime,
    getDayObsStartTime,
    makeEfdClient,
)
from lsst.ts.xml.enums.MTHexapod import ApplicationStatus, EnabledSubstate

In [ ]:
# Initialize an EFD client
efd_client = makeEfdClient()

# Global variables
HEXAPOD_AXES = ["X", "Y", "Z", "U", "V", "W"]
N_AXES = len(HEXAPOD_AXES)
N_STRUTS = 6

# We want to see all the data in a column
pd.set_option('display.max_colwidth', None)


# Enumerations for convenience
class CopleyStatusWord(enum.IntFlag):
    """
    Status word used in `mthexapod.electrical` as the `copleyStatusWordDrive` column. 
    See page 60 in the Copley Driver's Manual in the link below:

    - https://copleycontrols.com/wp-content/uploads/2018/02/All-CANopen_Programmers_Manual-Manual.pdf
    
    The numbers in the page above are the bit positions, and the values
    are the corresponding powers of 2. For example, the READY_TO_SWITCH_ON
    bit is at position 0, which corresponds to the value 2^0 = 1.
    The SWITCHED_ON bit is at position 1, which corresponds to the value
    2^1 = 2, and so on. 
    
    The values are powers of 2, so they can be combined
    using bitwise OR operations to represent multiple states at once. 
    These values are represented here as hexadecimal values for convenience,
    but they can also be used as decimal values.
    
    The values are also used in the `copleyStatusWordDrive` column in
    `mthexapod.electrical` to indicate the current status of the drive
    in the Copley hexapod system.
    """

    READY_TO_SWITCH_ON = 0x1
    SWITCHED_ON = 0x2
    OPERATION_ENABLED = 0x4
    FAULT = 0x8
    VOLTAGE_ENABLED = 0x10
    QUICK_STOP = 0x20
    SWITCH_ON_DISABLED = 0x40
    WARNING = 0x80
    LAST_TRAJECTORY_ABORTED = 0x100
    REMOTE_ON = 0x200
    TARGET_REACHED = 0x400
    INTERNAL_LIMIT_ACTIVE = 0x800
    SET_POINT_ACK = 0x1000
    FOLLOWING_ERROR = 0x2000
    MOVING = 0x4000
    CAPTURED_HOME_POSITION = 0x8000


class CopleyLatchingFaultStatus(enum.IntFlag):
    """
    Status word used in `mthexapod.electrical` as the `copleyLatchingFaultStatus` column. 
    See page 70 in the Copley Driver's Manual in the link below:

    - https://copleycontrols.com/wp-content/uploads/2018/02/All-CANopen_Programmers_Manual-Manual.pdf
    
    The numbers in the page above are the bit positions, and the values
    are the corresponding powers of 2. For example, the READY_TO_SWITCH_ON
    bit is at position 0, which corresponds to the value 2^0 = 1.
    The SWITCHED_ON bit is at position 1, which corresponds to the value
    2^1 = 2, and so on. 
    
    The values are powers of 2, so they can be combined
    using bitwise OR operations to represent multiple states at once. 
    These values are represented here as hexadecimal values for convenience,
    but they can also be used as decimal values.
    """
   
    FATAL_DATA_FLASH = 0x1
    FATAL_AMPLIFIER_INTERNAL_ERROR = 0x2
    SHORT_CIRCUIT = 0x4
    AMPLIFIER_OVER_TEMPERATURE = 0x8
    MOTOR_OVER_TEMPERATURE = 0x10
    OVER_VOLTAGE = 0x20
    UNDER_VOLTAGE = 0x40
    FEEDBACK_FAULT = 0x80
    PHASING_ERROR = 0x100
    TRACKING_ERROR = 0x200
    OVER_CURRENT = 0x400
    FPGA_FAILURE = 0x800
    INPUT_LOST = 0x1000
    FPGA_FAILURE_TWO = 0x2000
    SAFETY_CIRCUIT_FAULT = 0x4000
    UNABLE_TO_CONTROL_CURRENT = 0x8000


class CopleyFaultStatus(enum.IntFlag):
    """
    Status word used in `mthexapod.electrical` as the `copleyFaultStatus` column. 
    See page 62 in the Copley Driver's Manual in the link below:

    - https://copleycontrols.com/wp-content/uploads/2018/02/All-CANopen_Programmers_Manual-Manual.pdf
    
    The numbers in the page above are the bit positions, and the values
    are the corresponding powers of 2. For example, the READY_TO_SWITCH_ON
    bit is at position 0, which corresponds to the value 2^0 = 1.
    The SWITCHED_ON bit is at position 1, which corresponds to the value
    2^1 = 2, and so on. 
    
    The values are powers of 2, so they can be combined
    using bitwise OR operations to represent multiple states at once. 
    These values are represented here as hexadecimal values for convenience,
    but they can also be used as decimal values.
    """

    SHORT_CIRCUIT = 0x1
    AMPLIFIER_OVER_TEMPERATURE = 0x2
    OVER_VOLTAGE = 0x4
    UNDER_VOLTAGE = 0x8
    MOTOR_TEMPERATURE_SENSOR_ACTIVE = 0x10
    FEEDBACK_ERROR = 0x20
    MOTOR_PHASING_ERROR = 0x40
    CURRENT_OUTPUT_LIMITED = 0x80
    VOLTAGE_OUTPUT_LIMITED = 0x100
    POS_LIMIT_SWITCH_ACTIVE = 0x200
    NEG_LIMIT_SWITCH_ACTIVE = 0x400
    ENABLE_INPUT_NOT_ACTIVE = 0x800
    AMP_DISABLED_BY_SOFTWARE = 0x1000
    TRYING_TO_STOP_MOTOR = 0x2000
    MOTOR_BRAKE_ACTIVE = 0x4000
    PWM_OUTPUT_DISABLED = 0x8000
    POSITIVE_SW_LIMIT_CONDITION = 0x10000
    NEGATIVE_SW_LIMIT_CONDITION = 0x20000
    TRACKING_ERROR = 0x40000
    TRACKING_WARNING = 0x80000
    AMPLIFIER_IN_RESET_CONDITION = 0x100000
    POSITION_WRAPPED = 0x200000  # See the manual for more
    AMPLIFIER_FAULT = 0x400000
    REACHED_VELOCITY_LIMIT = 0x800000
    REACHED_ACCELERATION_LIMIT = 0x1000000
    POSITION_ERROR = 0x2000000
    HOME_SWITCH_IS_ACTIVE = 0x4000000
    IN_MOTION = 0x8000000
    VELOCITY_WINDOW = 0x10000000
    PHASE_NOT_YET_INITIALIZED = 0x20000000
    COMMAND_FAULT = 0x40000000


# Set global font size for labels, titles, and ticks
plt.rcParams.update(
    {
        "axes.grid": True,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "axes.formatter.useoffset": False,
        "axes.formatter.use_mathtext": False,
        "axes.formatter.limits": (-100, 100),
        "figure.figsize": (11, 6),
        "font.size": 12,
        "grid.color": "#b0b0b0",
        "grid.linestyle": ":",
        "grid.linewidth": 0.5,
        "grid.alpha": 0.75,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    }
)

level_colors = {
    "D": "#1f77b4",  # blue
    "I": "#2ca02c",  # green
    "W": "#ff7f0e",  # orange
    "E": "#d62728",  # red
    "C": "#9467bd",  # purple
}

## Data Analysis

Let's start defining some helper functions.  
For now, let me focus in querying and understanding the data.  
Remember that we want to know every time that the hexapod faulted and
find out which strut was possibly responsible for this fault.

In [ ]:
async def query_hexapod_controller_state(
    client: EfdClient, start_time: Time, end_time: Time, sal_index: int
) -> pd.DataFrame:
    """
    Query the hexapod controller state between `start_day_obs` and `end_day_obs`
    for a given `sal_index` (1: camera hexapod, 2: m2 hexapod)
    """
    query = f"""
        SELECT time, enabledSubstate, applicationStatus
        FROM "lsst.sal.MTHexapod.logevent_controllerState"
        WHERE time >= '{start_time.isot}Z'
        AND time <= '{end_time.isot}Z'
        AND salIndex = {sal_index}
    """

    _df = await client.influx_client.query(query)
    if not all(_df):
        print(
            "No data found for the specified time range and sal_index. "
            "Returning empty DataFrame."
            )
        return pd.DataFrame()

    _df["applicationStatusName"] = _df["applicationStatus"].apply(
        lambda x: ApplicationStatus(x).name
    )
    _df["enabledSubstateName"] = _df["enabledSubstate"].apply(
        lambda x: EnabledSubstate(x).name
    )

    return _df


async def query_hexapod_electrical(
    client: EfdClient, start_time: Time, end_time: Time, sal_index: int
) -> pd.DataFrame:

    columns = [
        f"copleyStatusWordDrive{i}, copleyLatchingFaultStatus{i}, copleyFaultStatus{i}" 
        for i in range(N_STRUTS)
    ]

    query = f"""
        SELECT {", ".join(columns)}
        FROM "lsst.sal.MTHexapod.electrical"
        WHERE time >= '{start_time.isot}Z'
        AND time <= '{end_time.isot}Z'
        AND salIndex = {sal_index}
        """
        
    _df = await client.influx_client.query(query)
    
    for i in range(N_STRUTS):
        _df[f"copleyStatusWordDrive{i}"] = _df[f"copleyStatusWordDrive{i}"].apply(
            lambda x: CopleyStatusWord(x) if x is not None else None
        )
        _df[f"copleyLatchingFaultStatus{i}"] = _df[f"copleyLatchingFaultStatus{i}"].apply(
            lambda x: CopleyLatchingFaultStatus(x) if x is not None else None
        )
        _df[f"copleyFaultStatus{i}"] = _df[f"copleyFaultStatus{i}"].apply(
            lambda x: CopleyFaultStatus(x) if x is not None else None
        )
    
    return _df


async def query_hexapod_log_messages(
    client: EfdClient, start_day_obs: int, end_day_obs: int, sal_index: int
) -> pd.DataFrame:
    """
    Query error messages from the log messages between `start_day_obs` and
    `end_day_obs` for a given `sal_index` (1: camera hexapod, 2: m2 hexapod)
    """
    start_time = getDayObsStartTime(start_day_obs)
    end_time = getDayObsEndTime(end_day_obs)

    query = f"""
        SELECT functionName, level, lineNumber, message
        FROM "lsst.sal.MTHexapod.logevent_logMessage"
        WHERE time >= '{start_time.isot}Z'
        AND time <= '{end_time.isot}Z'
        AND salIndex = {sal_index}
        AND level >= {min_log_level}
    """

    _df = await client.influx_client.query(query)

    return _df

Now, let me find out what is the size of my dataframe.  
This will affect how I will design the rest of my notebook.

In [ ]:
log_messages_df = await query_hexapod_log_messages(
    efd_client, start_day_obs, end_day_obs, sal_index
)

print(
    "The total number of log messages between {} and {} is {}".format(
        start_day_obs, end_day_obs, log_messages_df.index.size  
    )
)

In [ ]:
log_messages_df.tail(10)

It is quite a lot.  
I will start with the first error in the data frame and
I will print out a plot with the status for each of the columns in the electrical dataframe. 

In [ ]:
async def get_hexapod_data_around_log(
    log_messages_df : pd.DataFrame, 
    log_index : int, 
    sal_index: int,
    time_window: int=20 
) -> None:
    """
    Given a log_messages_df and a log_index, return controller state and electrical data
    around the log entry within the specified time window.
    """
    time_window = timedelta(seconds=time_window)
    time_reference = Time(log_messages_df.index[log_index])
    time_end = time_reference 
    time_start = time_reference - time_window

    controller_state_df = await query_hexapod_controller_state(
        efd_client, time_start, time_end, sal_index
    )

    electrical_data = await query_hexapod_electrical(
        efd_client, time_start, time_end, sal_index
    )
    
    return controller_state_df, electrical_data

In [ ]:
controller_state_df, electrical_data_df = await get_hexapod_data_around_log(
    log_messages_df, 0, sal_index, time_window=2
)

Let's have a look at our data.  
The events in `controller_state_df` are already decoded.  
This means that you can find out what is going on on each event.  

In [ ]:
controller_state_df.head(10)

The data in the `electrical_data_df` is not translated yet since I want to unpack the enumeration later. 

In [ ]:
electrical_data_df.head(5)

## Bitwise Operations

It is very hard to read the bits in a dataframe format.  
So let me try to plot them.  
This might make it easier to understand which strut failed first. 

Let's start with some useful functions that we can use in our analysis.  

In [ ]:
def decode_flags(value: int) -> List[CopleyStatusWord]:
    """
    Return the list of flags set in `value`.

    Examples
    --------
    >>> decode_flags(int(CopleyStatusWord.OPERATION_ENABLED | CopleyStatusWord.INTERNAL_LIMIT_ACTIVE))
    [
        <CopleyStatusWord.READY_TO_SWITCH_ON: 1>, 
        <CopleyStatusWord.SWITCHED_ON: 2>, 
        <CopleyStatusWord.OPERATION_ENABLED: 4>, 
        <CopleyStatusWord.VOLTAGE_ENABLED: 16>, 
        <CopleyStatusWord.REMOTE_ON: 512>, 
        <CopleyStatusWord.INTERNAL_LIMIT_ACTIVE: 2048>
    ]
    """
    flags = CopleyStatusWord(value)
    return [f for f in CopleyStatusWord if flags & f]


def has_flag(value: int, flag: CopleyStatusWord) -> bool:
    """
    True if `flag` is set in `value`.

    Examples
    --------
    >>> has_flag(int(CopleyStatusWord.INTERNAL_LIMIT_ACTIVE | CopleyStatusWord.OPERATION_ENABLED),
    ...          CopleyStatusWord.INTERNAL_LIMIT_ACTIVE)
    True
    >>> has_flag(0, CopleyStatusWord.FOLLOWING_ERROR)
    False
    """
    return bool(value & int(flag))


def struts_with_flag_in_row(
    row: pd.Series,
    flag: CopleyStatusWord,
    col_template: str = "copleyStatusWordDrive{}",
    drives: Iterable[int] = range(6),
    labels: List[str] | None = None,
) -> List[int | str]:
    """
    Given one DataFrame row, return drives (or labels) whose status word has `flag`.

    Parameters
    ----------
    row : pd.Series
        One row from your table.
    flag : CopleyStatusWord
        The flag to test (e.g. CopleyStatusWord.INTERNAL_LIMIT_ACTIVE).

    Examples
    --------
    >>> df = pd.DataFrame(
    ...     {
    ...         "copleyStatusWordDrive0": [0x800, 0],
    ...         "copleyStatusWordDrive1": [0, 0x800],
    ...         "copleyStatusWordDrive2": [0x800 | 0x2000, 0x2000],
    ...     },
    ...     index=pd.date_range("2025-01-01", periods=2, freq="S", tz="UTC"),
    ... )
    >>> row0 = df.iloc[0]
    >>> struts_with_flag_in_row(row0, CopleyStatusWord.INTERNAL_LIMIT_ACTIVE, drives=range(3))
    [0, 2]
    """
    bit = int(flag)
    out = []
    for i in drives:
        v = int(row.get(col_template.format(i), 0) or 0)
        if v & bit:
            out.append(labels[i] if labels else i)
    return out


def bit_active_df(
    df: pd.DataFrame,
    flag: CopleyStatusWord,
    drives: Iterable[int] = range(6),
    col_template: str = "copleyStatusWordDrive{}",
) -> pd.DataFrame:
    """
    Boolean table: rows=time, columns=drives, True when `flag` is active.

    Examples
    --------
    >>> df = pd.DataFrame(
    ...     {
    ...         "copleyStatusWordDrive0": [0x800, 0],
    ...         "copleyStatusWordDrive1": [0, 0x800],
    ...         "copleyStatusWordDrive2": [0x800 | 0x2000, 0x2000],
    ...     },
    ...     index=pd.date_range("2025-01-01", periods=2, freq="S", tz="UTC"),
    ... )
    >>> ila = bit_active_df(df, CopleyStatusWord.INTERNAL_LIMIT_ACTIVE, drives=range(3))
    >>> list(ila.columns)
    [0, 1, 2]
    >>> ila.iloc[0].to_dict()  # doctest: +ELLIPSIS
    {0: True, 1: False, 2: True}
    """
    bit = np.uint16(int(flag))
    cols = [col_template.format(i) for i in drives]
    sub = df[cols].fillna(0).astype("UInt16")
    active = (sub.values.astype(np.uint16) & bit) != 0
    return pd.DataFrame(active, index=df.index, columns=list(drives))


def list_struts_per_row(active_df: pd.DataFrame) -> pd.Series:
    """
    Convert a per-drive boolean DataFrame into a Series of lists per row.

    Examples
    --------
    >>> df = pd.DataFrame(
    ...     {
    ...         0: [True, False],
    ...         1: [False, True],
    ...         2: [True, True],
    ...     },
    ...     index=pd.date_range("2025-01-01", periods=2, freq="S", tz="UTC"),
    ... )
    >>> list_struts_per_row(df).tolist()
    [[0, 2], [1, 2]]
    """
    return active_df.apply(
        lambda r: [i for i, ok in zip(active_df.columns, r.values) if ok], axis=1
    )


def rising_edges(active_df: pd.DataFrame) -> pd.DataFrame:
    """
    Mark 0→1 transitions per drive (first instant a flag turns on).

    Examples
    --------
    >>> df = pd.DataFrame(
    ...     {
    ...         0: [False, True, True],
    ...         1: [False, False, True],
    ...     },
    ...     index=pd.date_range("2025-01-01", periods=3, freq="S", tz="UTC"),
    ... )
    >>> re_df = rising_edges(df)
    >>> re_df.iloc[0].to_dict()
    {0: False, 1: False}
    >>> re_df.iloc[1].to_dict()
    {0: True, 1: False}
    >>> re_df.iloc[2].to_dict()
    {0: False, 1: True}
    """
    prev = active_df.shift(1, fill_value=False)
    return (~prev) & active_df

The following cells show some examples on how to use the helper functions above.

In [ ]:
my_sample_bit = electrical_data_df.iloc[0].copleyStatusWordDrive0
print("My sample bit:", my_sample_bit)

# Decode the sample bit
decoded_flags = decode_flags(my_sample_bit)
print("Decoded flags:", decoded_flags)

# Does it have the INTERNAL_LIMIT_ACTIVE flag?
has_internal_limit_active = has_flag(my_sample_bit, CopleyStatusWord.INTERNAL_LIMIT_ACTIVE)
print("Has INTERNAL_LIMIT_ACTIVE flag:", has_internal_limit_active)

## Bitwise Display

In [ ]:
def ensure_timestamp_column(df: pd.DataFrame, time_col: str | None = None) -> pd.DataFrame:
    """
    Return a copy with a UTC datetime column 'ts', sorted by time.

    Examples
    --------
    >>> out = ensure_timestamp_column(pd.DataFrame({'t':['2025-01-01T00:00:00Z']}), 't')
    >>> 'ts' in out.columns and pd.api.types.is_datetime64tz_dtype(out['ts'])
    True
    """
    out = df.copy()
    if time_col is not None:
        out["ts"] = pd.to_datetime(out[time_col], utc=True, errors="coerce")
    elif isinstance(out.index, pd.DatetimeIndex):
        ts = out.index
        ts = ts.tz_localize("UTC") if ts.tz is None else ts.tz_convert("UTC")
        out["ts"] = ts
    else:
        parsed = None
        for c in out.columns:
            cand = pd.to_datetime(out[c], utc=True, errors="coerce")
            if cand.notna().any():
                parsed = cand; break
        if parsed is None:
            raise ValueError("No datetime column found; set `time_col`.")
        out["ts"] = parsed
    return out.dropna(subset=["ts"]).sort_values("ts")


def find_drive_columns(df: pd.DataFrame, drive_regex: str, limit: int) -> List[Tuple[int, str]]:
    """
    Find columns matching a drive bitfield; regex must capture the drive index in group 1.

    Examples
    --------
    >>> cols = ['copleyStatusWordDrive0','copleyStatusWordDrive2']
    >>> df = pd.DataFrame(columns=cols)
    >>> find_drive_columns(df, r'copleyStatusWordDrive(\\d+)$', 6)
    [(0, 'copleyStatusWordDrive0'), (2, 'copleyStatusWordDrive2')]
    """
    patt = re.compile(drive_regex, re.IGNORECASE)
    pairs = [(int(m.group(1)), c) for c in df.columns if (m := patt.search(c))]
    pairs.sort()
    return pairs[:limit]


def enum_members_bit_order(enum_cls: type[enum.IntFlag]) -> List[enum.IntFlag]:
    """
    Return enum members ordered by bit index (value.bit_length()-1).

    Examples
    --------
    >>> [m.name for m in enum_members_bit_order(CopleyStatusWord)][:3]
    ['READY_TO_SWITCH_ON', 'SWITCHED_ON', 'OPERATION_ENABLED']
    """
    return sorted(enum_cls, key=lambda m: int(m).bit_length() - 1)


def faultish_names(enum_cls: type[enum.IntFlag]) -> set[str]:
    """
    Heuristic to flag problematic bits; customize to your policy.

    Examples
    --------
    >>> 'FAULT' in faultish_names(CopleyStatusWord)
    True
    """
    keys = ("FAULT", "ERROR", "WARNING", "LIMIT", "ABORT")
    return {m.name for m in enum_cls if any(k in m.name for k in keys)}


def expand_bits_long_intflag(
    times: np.ndarray,
    bitfield_series: pd.Series,
    enum_cls: type[enum.IntFlag],
    faultish: set[str] | None = None,
) -> pd.DataFrame:
    """
    Expand a uint16 IntFlag series into long format: one row per (time, bit).
    Columns: ts, bit_idx, bit_name, active {0,1}, is_faultish {bool}.

    Examples
    --------
    >>> s = pd.Series([int(CopleyStatusWord.INTERNAL_LIMIT_ACTIVE)], index=pd.to_datetime(['2025-01-01'], utc=True))
    >>> long = expand_bits_long_intflag(s.index.to_numpy(), s, CopleyStatusWord)
    >>> long.query("active == 1 and bit_name == 'INTERNAL_LIMIT_ACTIVE'").shape[0] >= 1
    True
    """
    members = enum_members_bit_order(enum_cls)
    names = [m.name for m in members]
    vals = bitfield_series.fillna(0).astype("UInt16").to_numpy(np.uint16)
    bit_vals = np.array([int(m) for m in members], dtype=np.uint16)  # powers of 2
    # (n_rows, n_bits) -> active matrix
    B = (vals[:, None] & bit_vals[None, :]) != 0

    if faultish is None:
        faultish = faultish_names(enum_cls)
    is_fault_vec = np.array([name in faultish for name in names], dtype=bool)

    return pd.DataFrame({
        "ts":       np.repeat(times, len(members)),
        "bit_idx":  np.tile([int(m).bit_length() - 1 for m in members], len(vals)),
        "bit_name": np.tile(names, len(vals)),
        "active":   B.reshape(-1).astype(np.uint8),
        "is_faultish": np.tile(is_fault_vec, len(vals)),
    })


def estimate_rect_width_ms(times: Iterable[pd.Timestamp]) -> float:
    """
    Estimate rectangle width (milliseconds) from median cadence.

    Examples
    --------
    >>> idx = pd.date_range('2025-01-01', periods=3, freq='100L', tz='UTC')
    >>> 80 <= estimate_rect_width_ms(idx) <= 120
    True
    """
    s = pd.Series(times)
    if len(s) > 1:
        dt_ms = (s.diff().dt.total_seconds() * 1000).median()
        if pd.notna(dt_ms) and dt_ms > 0:
            return float(dt_ms)
    return 1000.0


# -------------------------- plotting pieces --------------------------
def make_drive_figure_colored(
    long_df: pd.DataFrame,
    y_labels: List[str],
    drive_idx: int,
    rect_width_ms: float,
    width: int,
    height: int,
    shared_x=None,
):
    """
    One drive panel drawn with 3 layers:
      - inactive -> lightgray
      - active & fault-ish -> firebrick
      - active & normal -> steelblue

    Examples
    --------
    >>> df_long = pd.DataFrame({'ts': pd.date_range('2025-01-01', periods=2, tz='UTC').repeat(2),
    ...                         'bit_name':['A','B']*2, 'active':[1,0,0,1], 'is_faultish':[True,False,True,False]})
    >>> fig = make_drive_figure_colored(df_long, ['A','B'], 0, 1000, 400, 180)
    >>> fig.x_axis_type == 'datetime'
    True
    """
    p = figure(
        x_axis_type="datetime",
        y_range=FactorRange(*y_labels),
        width=width, height=height,
        title=f"COPLEY STATUS WORDS – Drive {drive_idx}",
        tools="xpan,xwheel_zoom,reset,save,hover",
        active_scroll="xwheel_zoom",
    )
    if shared_x is not None:
        p.x_range = shared_x

    # pre-format ms timestamp for hover (portable across Bokeh versions)
    ldf = long_df.copy()
    ldf["ts_ms"] = pd.to_datetime(ldf["ts"]).dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]
    src = ColumnDataSource(ldf)

    inactive_mask     = (ldf["active"] == 0).tolist()
    active_fault_mask = ((ldf["active"] == 1) & (ldf["is_faultish"] == True)).tolist()
    active_ok_mask    = ((ldf["active"] == 1) & (ldf["is_faultish"] == False)).tolist()

    view_inactive = CDSView(filter=BooleanFilter(booleans=inactive_mask))
    view_fault    = CDSView(filter=BooleanFilter(booleans=active_fault_mask))
    view_ok       = CDSView(filter=BooleanFilter(booleans=active_ok_mask))

    r_inactive = p.rect(x="ts", y="bit_name", width=rect_width_ms, height=0.9,
                        source=src, view=view_inactive, line_color=None,
                        fill_color="lightgray", fill_alpha=0.35)

    r_fault = p.rect(x="ts", y="bit_name", width=rect_width_ms, height=0.9,
                     source=src, view=view_fault, line_color=None,
                     fill_color="firebrick", fill_alpha=0.95)

    r_ok = p.rect(x="ts", y="bit_name", width=rect_width_ms, height=0.9,
                  source=src, view=view_ok, line_color=None,
                  fill_color="steelblue", fill_alpha=0.95)

    # legend outside plot area
    legend = Legend(items=[
        ("inactive", [r_inactive]),
        ("active (fault-ish)", [r_fault]),
        ("active (normal)", [r_ok]),
    ], orientation="vertical", click_policy="hide")
    p.add_layout(legend, "right")

    hv = p.select_one(HoverTool) or HoverTool()
    if hv not in p.tools: p.add_tools(hv)
    hv.tooltips = [("time", "@ts_ms"), ("bit", "@bit_name"), ("active", "@active"), ("fault-ish", "@is_faultish")]

    p.yaxis.axis_label = "Flags"
    p.xaxis.axis_label = "Timestamp (UTC)"
    return p


def arrange_figures_column(figs: List, *, toolbar_location: str = "above"):
    """
    Arrange figures in a single column (one plot per row).

    Examples
    --------
    >>> layout = arrange_figures_column([])
    >>> hasattr(layout, 'children')
    True
    """
    return gridplot(figs, ncols=1, toolbar_location=toolbar_location, merge_tools=True)


def add_synced_vertical_cursor(figs: List, *, line_width=1, line_alpha=0.85):
    """
    Add a shared vertical cursor (Span) that moves together across all figures.

    Examples
    --------
    >>> p1 = figure(x_axis_type='datetime'); p2 = figure(x_axis_type='datetime')
    >>> add_synced_vertical_cursor([p1, p2])
    """
    spans = [Span(location=None, dimension="height", line_width=line_width, line_alpha=line_alpha) for _ in figs]
    for p, s in zip(figs, spans):
        p.add_layout(s)
        p.js_on_event('mousemove', CustomJS(args=dict(spans=spans), code="""
            const x = cb_obj.x; for (const sp of spans) { sp.location = x; }
        """))
        p.js_on_event('mouseleave', CustomJS(args=dict(spans=spans), code="""
            for (const sp of spans) { sp.location = null; }
        """))

# -------------------------- main builder --------------------------
def copley_status_heatmaps_bokeh_intflag(
    df: pd.DataFrame,
    *,
    time_col: str | None = None,
    drive_regex: str = r"copleyStatusWordDrive(\d+)$",
    enum_cls: type[enum.IntFlag] = CopleyStatusWord,
    faultish: set[str] | None = None,
    n_axes: int = 6,
    width: int = 900,
    height: int = 250,
):
    """
    Build linked Bokeh heatmaps (one per drive) for IntFlag bitfields.
    Single column, legend outside, synced cursor, ms in hover.

    Examples
    --------
    >>> # df must have a datetime index/column and copleyStatusWordDrive0..5
    >>> # from bokeh.io import output_notebook, show
    >>> # output_notebook()
    >>> # layout = copley_status_heatmaps_bokeh_intflag(df, time_col='timestamp')
    >>> # show(layout)
    """
    df_ts = ensure_timestamp_column(df, time_col)
    drive_cols = find_drive_columns(df_ts, drive_regex, limit=n_axes)
    if not drive_cols:
        raise ValueError(f"No drive columns match regex: {drive_regex}")

    members = enum_members_bit_order(enum_cls)
    y_labels = [m.name for m in members]
    rect_w = estimate_rect_width_ms(df_ts["ts"])

    figs, shared_x = [], None
    for i, (drive_idx, col) in enumerate(drive_cols):
        long_df = expand_bits_long_intflag(df_ts["ts"].to_numpy(), df_ts[col], enum_cls, faultish=faultish)
        fig = make_drive_figure_colored(
            long_df=long_df,
            y_labels=y_labels,
            drive_idx=drive_idx,
            rect_width_ms=rect_w,
            width=width,
            height=height,
            shared_x=shared_x,
        )
        if shared_x is None:
            shared_x = fig.x_range
        else:
            fig.x_range = shared_x
        figs.append(fig)

    add_synced_vertical_cursor(figs)
    return arrange_figures_column(figs)


In [ ]:
from bokeh.io import output_notebook, show

output_notebook()
layout = copley_status_heatmaps_bokeh_intflag(electrical_data_df)
show(layout)

## Explore the data

I wanted to find an easy way to explore all the faults.  
It ended up being too hard.  
So here is a cheated version.  
Use the date picker and select a row from the dropdown menu.
  
Once you are happy with the fault you want to select,  
run the other two cells.  
  
They will query the data in a new dataframe and display the data.  
Be aware that errors containing "Compensation failed" won't work in this notebook. 

In [ ]:
def make_log_row_picker_utc(
    df: pd.DataFrame,
    *,
    id_cols: tuple[str, ...] = ("message", "msg", "text", "detail", "summary"),
    label_max: int = 120,
):
    """
    Interactive UTC picker for a log DataFrame whose index is a DatetimeIndex (assumed UTC).
    - Pick a day (UTC)
    - Pick a row
    - Prints the selected row's `.iloc` positional index

    Returns
    -------
    dict with:
      - 'view': VBox widget to display
      - 'state': SimpleNamespace(selected_ts, selected_pos)
      - handles: 'date', 'dropdown', 'info', 'out'
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("df.index must be a DatetimeIndex (assumed UTC).")

    # Work in UTC-naive timestamps for filtering (no timezone ops)
    if df.index.tz is not None:
        idx_naive = df.index.tz_convert("UTC").tz_localize(None)
    else:
        idx_naive = df.index

    # Widgets
    dp = w.DatePicker(description="Date (UTC)")
    dd = w.Dropdown(description="Row", options=[("— pick a date —", None)], value=None, layout=w.Layout(width="95%"))
    info = w.HTML(layout=w.Layout(width="100%"))
    out = w.Output()
    state = SimpleNamespace(selected_ts=None, selected_pos=None)

    # Helpers
    def _summary_from_row(row: pd.Series) -> str:
        for c in id_cols:
            if c in row and pd.notna(row[c]):
                s = str(row[c]);  return s if len(s) <= label_max else (s[: label_max - 1] + "…")
        for c in df.columns:
            v = row.get(c, None)
            if pd.notna(v):
                s = str(v);  return s if len(s) <= label_max else (s[: label_max - 1] + "…")
        return "(no details)"

    def _format_row_html(ts: pd.Timestamp, row: pd.Series) -> str:
        # ts expected UTC-naive; pretty print with ms
        head = f"<b>{ts.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]} UTC</b>"
        rows = "".join(
            f"<tr><th style='text-align:left;padding-right:8px'>{escape(str(k))}</th>"
            f"<td style='text-align:left'>{escape(str(row[k]))}</td></tr>"
            for k in df.columns
        )
        return f"{head}<table style='margin-top:6px'>{rows}</table>"

    def _iloc_from_timestamp(ts_original_index: pd.Timestamp) -> int | None:
        """Map the original timestamp in df.index -> integer iloc (first if duplicated)."""
        try:
            loc = df.index.get_loc(ts_original_index)
        except KeyError:
            return None
        if isinstance(loc, slice):
            return loc.start
        try:
            return int(loc[0])
        except Exception:
            return int(loc)

    # Callbacks
    def refresh_dropdown(*_):
        info.value = ""; out.clear_output()
        if dp.value is None:
            dd.options = [("— pick a date —", None)]
            dd.value = None
            return
        start = pd.Timestamp(dp.value)                    # UTC midnight (naive)
        end   = start + pd.Timedelta(days=1)
        mask = (idx_naive >= start) & (idx_naive < end)
        sub = df.loc[mask]
        if sub.empty:
            dd.options = [("— no messages —", None)]
            dd.value = None
            return

        # Build (label, original_index_ts) options; label uses UTC-naive time w/ ms
        opts = []
        # Use the *original* index for selection, but format using idx_naive for display
        sub_idx_naive = idx_naive[mask]
        for (orig_ts, row), disp_ts in zip(sub.iterrows(), sub_idx_naive):
            label = f"{disp_ts.strftime('%H:%M:%S.%f')[:-3]} — {_summary_from_row(row)}"
            opts.append((label, orig_ts))
        dd.options = opts
        dd.value = opts[0][1]  # triggers on_select

    def on_select(change):
        ts = change["new"]
        state.selected_ts = ts
        out.clear_output()
        if ts is None:
            info.value = ""
            state.selected_pos = None
            return

        # For display, convert to UTC-naive if needed
        ts_disp = ts.tz_convert("UTC").tz_localize(None) if getattr(ts, "tzinfo", None) is not None else ts
        row = df.loc[ts]
        info.value = _format_row_html(ts_disp, row)

        pos = _iloc_from_timestamp(ts)
        state.selected_pos = pos
        with out:
            if pos is None:
                print("No iloc position found.")
            else:
                print("Selected iloc index:", pos)

    # Wire
    dp.observe(refresh_dropdown, names="value")
    dd.observe(on_select, names="value")

    # Initialize to first available day in UTC
    if len(idx_naive):
        dp.value = idx_naive.min().date()

    return {"view": w.VBox([dp, dd, info, out]),
            "state": state, "date": dp, "dropdown": dd, "info": info, "out": out}


In [ ]:
ui = make_log_row_picker_utc(log_messages_df)
display(ui["view"])

In [ ]:
from bokeh.io import output_notebook, show


print("Selected log message position:", ui["state"].selected_pos)
controller_state_df, electrical_data_df = await get_hexapod_data_around_log(
    log_messages_df, ui["state"].selected_pos, sal_index, time_window=30
)

output_notebook()
layout = copley_status_heatmaps_bokeh_intflag(electrical_data_df)
show(layout)